In [ ]:
import csv
import concurrent.futures
import datetime
import glob
import gc
import json
import math
import multiprocessing
import os
import random
import re
import torch

import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import multiprocessing as mp

from unsloth import FastLanguageModel

from accelerate import Accelerator, PartialState
from collections import Counter, defaultdict, deque
from concurrent.futures import ProcessPoolExecutor, as_completed
from datasets import Dataset, concatenate_datasets, load_from_disk, IterableDataset, load_dataset
from datetime import timedelta
from imblearn.under_sampling import TomekLinks, NearMiss
from functools import partial
from math import inf
from multiprocessing import cpu_count, Pool
from pathlib import Path
from pandas import Timedelta
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType, PeftModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
from scipy import stats
from scipy.spatial import KDTree
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    TrainingArguments, DataCollatorForLanguageModeling,
    TrainerCallback
)
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
from trl import SFTConfig, SFTTrainer, DPOTrainer, DPOConfig
from typing import List, Dict, Optional, Tuple, Union, Set

In [ ]:
# CUDA / tokenizer runtime settings.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
# =========================
# Repository path settings
# =========================
PROJECT_ROOT = Path.cwd()

OUTPUT_ROOT = PROJECT_ROOT / "output"

# Change this value when tuning a different disk model.
# Examples:
#   Backblaze: "ST12000NM0008"
#   Alibaba Disk: "model-one" or "model-two"
DISK_MODEL_ID = "model-two"

MONTHLY_TEXTUAL_DIR = OUTPUT_ROOT / "data" / DISK_MODEL_ID / "monthly_textual"
MODEL_OUTPUT_DIR = OUTPUT_ROOT / "models" / DISK_MODEL_ID
CHECKPOINT_DIR = MODEL_OUTPUT_DIR / "checkpoints"
TRAINER_OUTPUT_DIR = MODEL_OUTPUT_DIR / "trainer_runs"
RESULT_DIR = OUTPUT_ROOT / "results"

METRICS_JSON_PATH = RESULT_DIR / f"{DISK_MODEL_ID}_tuning_metrics.json"
METRICS_CSV_PATH = RESULT_DIR / f"{DISK_MODEL_ID}_tuning_metrics.csv"

for path in [MODEL_OUTPUT_DIR, CHECKPOINT_DIR, TRAINER_OUTPUT_DIR, RESULT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Monthly textual input directory: {MONTHLY_TEXTUAL_DIR}")
print(f"Model output directory          : {MODEL_OUTPUT_DIR}")
print(f"Result directory                : {RESULT_DIR}")

## Helper Methods

In [ ]:
# =========================
# Prompt settings
# =========================
SYSTEM_PROMPT = (
    "You are an expert in hard drive failure prediction. "
    "Analyze the provided disk report and respond with the word either 'Yes' or 'No'. "
    "Do not include any explanations or additional text."
)

USER_INSTRUCTION_TEMPLATE = (
    "Please analyze the disk report: \n{description}\n\n"
    "Only some of the collected SMART attributes are shown in the report. "
    "Will this hard drive likely fail within the next 7 days and therefore warrant replacement? "
    "Answer either 'Yes' or 'No'. "
)

In [ ]:
def remove_failure_line(text: str) -> str:
    lines = text.splitlines()
    cleaned_lines = [
        line
        for line in lines
        if not line.strip().startswith("Failure in next 7 days:")
    ]
    return "\n".join(cleaned_lines)


def preprocess_description(example):
    desc = example.get("description", "")
    failure = 0

    if not isinstance(desc, str):
        return {
            "failure": 0,
            "description": "",
        }

    # Extract label before removing the leakage line.
    if "Failure in next 7 days: Yes" in desc:
        failure = 1

    cleaned_desc = remove_failure_line(desc)

    return {
        "failure": failure,
        "description": cleaned_desc,
    }


def clean_dataset(raw_ds, balance_ratio=6, seed=42):
    if isinstance(raw_ds, Dataset):
        data = raw_ds.to_pandas().copy()
    elif isinstance(raw_ds, list):
        dfs = []
        for f in raw_ds:
            df = pd.read_csv(f)
            required = {"serial_number", "failure", "description"}
            if not required.issubset(df.columns):
                raise ValueError(f"File {f} missing required columns: {required}")
            dfs.append(df)
        data = pd.concat(dfs, ignore_index=True)
    elif isinstance(raw_ds, pd.DataFrame):
        data = raw_ds.copy()
        required = {"serial_number", "failure", "description"}
        if not required.issubset(data.columns):
            raise ValueError(f"DataFrame must contain: {required}")
    else:
        raise TypeError("raw_ds must be a file list, pandas DataFrame, or Hugging Face Dataset")

    data["failure"] = data["failure"].astype(int)

    pos_df = data[data["failure"] == 1].copy()
    neg_df = data[data["failure"] == 0].copy()

    if len(pos_df) == 0:
        print("   No positive rows found in this month; returning shuffled original dataset.")
        return Dataset.from_pandas(
            data.sample(frac=1, random_state=seed).reset_index(drop=True),
            preserve_index=False,
        )

    failing_serials = set(pos_df["serial_number"].unique())

    neg_from_failing_disks = neg_df[neg_df["serial_number"].isin(failing_serials)].copy()
    neg_from_healthy_disks = neg_df[~neg_df["serial_number"].isin(failing_serials)].copy()

    target_neg = min(len(neg_df), balance_ratio * len(pos_df))
    target_neg_failing = target_neg // 2
    target_neg_healthy = target_neg - target_neg_failing

    sampled_neg_failing = neg_from_failing_disks.sample(
        n=min(target_neg_failing, len(neg_from_failing_disks)),
        random_state=seed,
    ) if len(neg_from_failing_disks) > 0 else neg_from_failing_disks

    sampled_neg_healthy = neg_from_healthy_disks.sample(
        n=min(target_neg_healthy, len(neg_from_healthy_disks)),
        random_state=seed + 1,
    ) if len(neg_from_healthy_disks) > 0 else neg_from_healthy_disks

    cleaned_data = pd.concat(
        [pos_df, sampled_neg_failing, sampled_neg_healthy],
        ignore_index=True,
    ).sample(frac=1, random_state=seed).reset_index(drop=True)

    print(
        f"   clean_dataset -> pos={len(pos_df)} | "
        f"neg_failing={len(sampled_neg_failing)} | "
        f"neg_healthy={len(sampled_neg_healthy)} | "
        f"total={len(cleaned_data)}"
    )

    return Dataset.from_pandas(cleaned_data, preserve_index=False)


def create_balanced_sample(ds, seed=42):
    random.seed(seed)

    if "serial_number" not in ds.column_names:
        raise ValueError("Dataset must have a serial_number column")

    serials = set(ds["serial_number"])
    failing_serials = {
        example["serial_number"]
        for example in ds
        if int(example["failure"]) == 1
    }

    num_failing = len(failing_serials)
    print(f"   Total unique disks: {len(serials)}")
    print(f"   Failing disks     : {num_failing}")

    healthy_serials = list(serials - failing_serials)
    num_failing_to_use = min(num_failing, MAX_EVAL_DISKS_PER_CLASS)

    sampled_failing_serials = random.sample(list(failing_serials), num_failing_to_use)

    if num_failing_to_use == 0:
        print("   No failing disks found; sampling healthy disks as fallback.")
        num_healthy_to_sample = 100
    else:
        num_healthy_to_sample = num_failing_to_use

    sampled_healthy_serials = random.sample(
        healthy_serials,
        min(num_healthy_to_sample, len(healthy_serials)),
    )

    failing_ds = ds.filter(lambda x: x["serial_number"] in sampled_failing_serials)
    healthy_ds = ds.filter(lambda x: x["serial_number"] in sampled_healthy_serials)

    sampled = concatenate_datasets([failing_ds, healthy_ds]).shuffle(seed=seed)
    pos_count = sampled.filter(lambda x: int(x["failure"]) == 1).num_rows

    print(
        f"   Eval sample: {len(sampled)} rows, "
        f"{pos_count} positives, "
        f"{len(sampled_failing_serials)} failing disks, "
        f"{len(sampled_healthy_serials)} healthy disks"
    )

    return sampled

In [ ]:
def build_fallback_prompt(description: str) -> str:
    user_content = USER_INSTRUCTION_TEMPLATE.format(description=description)
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{user_content}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )


def prepare_example(example):
    description = str(example.get("description", "")).strip()
    if not description:
        description = "No abnormal SMART attributes reported."

    user_instruction = USER_INSTRUCTION_TEMPLATE.format(description=description)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_instruction},
    ]

    try:
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception as e:
        print(f"Warning: using fallback template due to {e}")
        prompt = build_fallback_prompt(description)

    if int(example["failure"]) == 0:
        chosen = "No"
        rejected = "Yes"
    else:
        chosen = "Yes"
        rejected = "No"

    return {
        "prompt": prompt,
        "chosen": chosen,
        "rejected": rejected,
    }

In [ ]:
def get_mispredicted_dataset(model, tokenizer, base_ds, chunk_size=28):
    model.eval()

    original_padding_side = tokenizer.padding_side

    yes_tokens = tokenizer("Yes", add_special_tokens=False).input_ids
    no_tokens = tokenizer("No", add_special_tokens=False).input_ids

    if not yes_tokens or not no_tokens:
        raise ValueError("Tokenizer cannot properly tokenize 'Yes' or 'No'")

    first_yes_token = int(yes_tokens[0])
    first_no_token = int(no_tokens[0])
    allowed_first_tokens = [first_yes_token, first_no_token]

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    predictions = []
    total = len(base_ds)
    print(f"Starting misprediction inference on {total} examples...")

    try:
        tokenizer.padding_side = "left"

        for start_idx in range(0, total, chunk_size):
            end_idx = min(start_idx + chunk_size, total)
            chunk = base_ds.select(range(start_idx, end_idx))

            prompts = []
            for example in chunk:
                description = str(example.get("description", "")).strip()
                if not description:
                    description = "No abnormal SMART attributes reported."

                user_content = USER_INSTRUCTION_TEMPLATE.format(description=description)
                messages = [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_content},
                ]

                try:
                    full_prompt = tokenizer.apply_chat_template(
                        messages,
                        tokenize=False,
                        add_generation_prompt=True,
                    )
                except Exception as e:
                    print(f"Warning: apply_chat_template failed; using fallback ({e})")
                    full_prompt = build_fallback_prompt(description)

                prompts.append(full_prompt)

            inputs = tokenizer(
                prompts,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt",
            ).to(model.device)

            input_seq_len = int(inputs["input_ids"].shape[1])

            def prefix_allowed_tokens_fn(batch_id, input_ids):
                if len(input_ids) == input_seq_len:
                    return allowed_first_tokens
                return [tokenizer.eos_token_id]

            with torch.no_grad():
                generated_ids = model.generate(
                    **inputs,
                    max_new_tokens=1,
                    do_sample=False,
                    num_beams=1,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                    prefix_allowed_tokens_fn=prefix_allowed_tokens_fn,
                    return_dict_in_generate=False,
                )

            batch_preds = []
            for row_idx in range(generated_ids.shape[0]):
                new_ids = generated_ids[row_idx, input_seq_len:]

                if new_ids.numel() == 0:
                    pred = 0
                    print("  Warning: no generated binary token, defaulting to 'No'")
                else:
                    first_token = int(new_ids[0].item())
                    if first_token == first_yes_token:
                        pred = 1
                    elif first_token == first_no_token:
                        pred = 0
                    else:
                        pred = 0
                        raw_text = tokenizer.decode(new_ids[:8], skip_special_tokens=False)
                        print(f"  Unexpected binary token id={first_token}; decoded='{raw_text[:60]}'")

                batch_preds.append(pred)

            predictions.extend(batch_preds)

            del inputs, generated_ids
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

            if (start_idx // chunk_size + 1) % 10 == 0 or end_idx == total:
                print(f"  Processed {end_idx}/{total} examples...")
    finally:
        tokenizer.padding_side = original_padding_side

    base_ds = base_ds.add_column("prediction", predictions)

    tp = sum(1 for ex in base_ds if int(ex["failure"]) == 1 and int(ex["prediction"]) == 1)
    tn = sum(1 for ex in base_ds if int(ex["failure"]) == 0 and int(ex["prediction"]) == 0)
    fp = sum(1 for ex in base_ds if int(ex["failure"]) == 0 and int(ex["prediction"]) == 1)
    fn = sum(1 for ex in base_ds if int(ex["failure"]) == 1 and int(ex["prediction"]) == 0)

    total = len(base_ds)
    accuracy = (tp + tn) / total if total > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

    metrics = {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "total": total,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "fpr": fpr,
    }

    print(
        f"  Eval Metrics: TP={tp}, TN={tn}, FP={fp}, FN={fn} | "
        f"Acc={accuracy:.3f} | Prec={precision:.3f} | "
        f"Rec={recall:.3f} | F1={f1:.3f} | FPR={fpr:.3f}"
    )

    mispred_ds = base_ds.filter(lambda ex: int(ex["prediction"]) != int(ex["failure"]))

    columns_to_keep = ["description", "failure", "date", "serial_number"]
    columns_to_remove = [col for col in mispred_ds.column_names if col not in columns_to_keep]
    if columns_to_remove:
        mispred_ds = mispred_ds.remove_columns(columns_to_remove)

    print(f"  Completed: found {len(mispred_ds)} mispredicted hard examples")

    return mispred_ds, metrics


In [ ]:
def check_model_collapse(
    metrics,
    recall_floor=0.85,
    precision_floor=0.35,
    fpr_ceiling=0.50,
    fp_to_tp_ceiling=3.0,
):
    precision = metrics.get("precision", 0.0)
    recall = metrics.get("recall", 0.0)
    fpr = metrics.get("fpr", 0.0)
    tp = metrics.get("tp", 0)
    fp = metrics.get("fp", 0)

    positive_ratio = (tp + fp) / max(metrics.get("total", 1), 1)
    metrics["positive_prediction_ratio"] = positive_ratio

    if tp == 0 and fp == 0:
        return True, "no positive predictions"
    if recall < recall_floor:
        return True, "recall collapsed"
    if precision < precision_floor and recall >= 0.90:
        return True, "high-recall low-precision over-alerting"
    if fpr > fpr_ceiling:
        return True, "extreme false positive rate"
    if tp > 0 and (fp / (tp + 1e-8)) > fp_to_tp_ceiling:
        return True, "fp overwhelms tp"
    if positive_ratio > 0.65 and precision < 0.40:
        return True, "too many positive predictions"

    return False, "ok"


def create_balanced_replay(
    fn_examples,
    fp_examples,
    current_ds,
    seed,
    max_size=8000,
    fallback_ratio=0.5,
    include_healthy_fallback=True,
):
    if len(fn_examples) == 0 and len(fp_examples) == 0:
        print("   No mispredictions; using random sample from current month")
        return current_ds.shuffle(seed=seed).select(range(min(1500, len(current_ds))))

    parts = []

    if len(fn_examples) > 0:
        parts.append(fn_examples)
        print(f"   Replay keeps all FN: {len(fn_examples)}")

    if len(fp_examples) > 0:
        target_fp = min(
            len(fp_examples),
            max(100, int(len(fn_examples) * fallback_ratio)) if len(fn_examples) > 0 else 300,
        )
        fp_sample = fp_examples.shuffle(seed=seed).select(range(target_fp))
        parts.append(fp_sample)
        print(f"   Replay keeps sampled FP: {len(fp_sample)}")

    if include_healthy_fallback:
        healthy_from_current = current_ds.filter(lambda x: int(x["failure"]) == 0)
        if len(healthy_from_current) > 0:
            n_healthy = min(
                len(healthy_from_current),
                max(300, len(fn_examples)) if len(fn_examples) > 0 else 500,
            )
            healthy_sample = healthy_from_current.shuffle(seed=seed + 123).select(range(n_healthy))
            parts.append(healthy_sample)
            print(f"   Replay adds healthy fallback: {len(healthy_sample)}")

    if not parts:
        return current_ds.shuffle(seed=seed).select(range(min(1500, len(current_ds))))

    replay_ds = concatenate_datasets(parts).shuffle(seed=seed)

    if len(replay_ds) > max_size:
        replay_ds = replay_ds.select(range(max_size))

    return replay_ds

## Load Base Model

In [ ]:
model_tuning_start_time = time.perf_counter()

In [ ]:
MODEL_ID = "microsoft/phi-4"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if getattr(tokenizer, "pad_token_id", None) is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_storage=torch.uint8,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    use_cache=False,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_rslora=True,
    inference_mode=False,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Tuning Configuration

In [ ]:
MAX_REPLAY_SIZE = 8000
MAX_EVAL_DISKS_PER_CLASS = 500
REPLAY_CAP_PER_MONTH = 6000
POS_NEG_RATIO = 6
SEED_BASE = 42

# Number of monthly files used for tuning.
# For the paper protocol, this is the 15-month development period.
NUM_TUNING_MONTHS = 15

# Adaptive tuning parameters.
BETA_MIN = 0.03
BETA_MAX = 0.10
LR_MIN = 5e-7
LR_MAX = 2e-6
LABEL_SMOOTHING_MIN = 0.05
LABEL_SMOOTHING_MAX = 0.15
MAX_STEPS_MIN = 100
MAX_STEPS_MAX = 200

PRECISION_FLOOR = 0.35
RECALL_FLOOR = 0.85
FPR_CEILING = 0.50

prev_replay_size = None
metrics_history = []
replay_ds = None
prev_fn_examples = None
prev_fp_examples = None

current_beta = 0.05
current_lr = 1e-6
current_label_smoothing = 0.10
current_max_steps = 150

In [ ]:
dpo_config = DPOConfig(
    output_dir=str(TRAINER_OUTPUT_DIR),
    max_steps=200,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    beta=0.05,
    loss_type="sigmoid",
    label_smoothing=0.05,
    max_length=500,
    max_prompt_length=468,
    gradient_checkpointing=True,
    logging_steps=10,
    save_strategy="no",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine_with_restarts",
    bf16=True,
    optim="adamw_8bit",
    report_to="none",
    remove_unused_columns=False,
    dataset_num_proc=4,
)

In [ ]:
class EarlyStoppingCallback(TrainerCallback):
    def __init__(self, window_size=50, threshold=0.005, min_steps=100):
        self.window_size = window_size
        self.threshold = threshold
        self.min_steps = min_steps
        self.loss_history = []
        self.should_stop = False

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None or "loss" not in logs:
            return

        self.loss_history.append(logs["loss"])

        if len(self.loss_history) < self.min_steps:
            return

        recent_losses = self.loss_history[-self.window_size:]
        if len(recent_losses) < self.window_size:
            return

        loss_std = np.std(recent_losses)
        loss_change = abs(recent_losses[-1] - recent_losses[0])

        if loss_std < self.threshold and loss_change < self.threshold:
            print(
                f"Early stopping: loss stabilized "
                f"(std={loss_std:.6f}, change={loss_change:.6f})"
            )
            self.should_stop = True
            control.should_training_stop = True

In [ ]:
def extract_month_sort_key(path: Path):
    name = path.name

    # Matches 2024-01, 202401, or date ranges such as 2024-01-01_to_2024-01-31.
    match = re.search(r"(\d{4})[-_]?([01]\d)", name)
    if match:
        return f"{match.group(1)}-{match.group(2)}"

    return name


csv_files = sorted(MONTHLY_TEXTUAL_DIR.glob("*.csv"), key=extract_month_sort_key)

if not csv_files:
    raise FileNotFoundError(f"No monthly textual CSV files found under: {MONTHLY_TEXTUAL_DIR}")

csv_files = csv_files[:NUM_TUNING_MONTHS]

print(f"Found {len(csv_files)} monthly files for tuning")
print("Processing order:")
for file_path in csv_files:
    print(f"  - {file_path.name}")

## Run Tuning

In [ ]:
for i, csv_file in enumerate(csv_files):
    print(f"\n=== Processing month {i+1}/{len(csv_files)}: {os.path.basename(csv_file)} ===")

    collapse_detected = False
    collapse_reason = ""

    if metrics_history:
        collapse_detected, collapse_reason = check_model_collapse(metrics_history[-1])

    if collapse_detected:
        print(f"   COLLAPSE DETECTED ({collapse_reason}) - activating recovery mode")

        current_beta = 0.18

        if prev_fn_examples is not None:
            fn_to_use = prev_fn_examples
            fp_to_use = prev_fp_examples if prev_fp_examples is not None else None

            if fp_to_use is not None and len(fp_to_use) > 0:
                n_fp_keep = min(len(fp_to_use), max(200, len(fn_to_use) // 5))
                fp_sample = fp_to_use.shuffle(seed=SEED_BASE + i).select(range(n_fp_keep))
                replay_ds = concatenate_datasets([fn_to_use, fp_sample]).shuffle(seed=SEED_BASE + i)
                print(f"   Recovery replay: {len(fn_to_use)} FN + {n_fp_keep} FP from previous month")
            else:
                healthy_fallback = current_ds.filter(lambda x: x["failure"] == 0)
                if len(healthy_fallback) > 0:
                    n_fallback = min(len(fn_to_use), len(healthy_fallback))
                    fallback_sample = healthy_fallback.shuffle(seed=SEED_BASE + i).select(range(n_fallback))
                    replay_ds = concatenate_datasets([fn_to_use, fallback_sample]).shuffle(seed=SEED_BASE + i)
                    print(f"   Recovery replay: {len(fn_to_use)} FN + {n_fallback} healthy fallback")
                else:
                    replay_ds = fn_to_use
        else:
            print("     No previous mispredictions available for recovery")

        temp_replay_cap = min(15000, MAX_REPLAY_SIZE * 1.5)
        print(f"   Recovery settings → beta={current_beta:.3f} | replay cap={temp_replay_cap}")

    else:
        if i < 3:
            current_beta = 0.20
        elif i < 8:
            current_beta = 0.20
        else:
            current_beta = 0.18

        temp_replay_cap = REPLAY_CAP_PER_MONTH
        print(f"   Normal mode → beta={current_beta:.3f}")

    dpo_config.beta = current_beta

    current_ds = load_dataset("csv", data_files=[csv_file])["train"]

    current_ds = current_ds.map(
        preprocess_description,
        desc="Labeling failures & removing leakage"
    )

    current_ds = clean_dataset(current_ds, balance_ratio=POS_NEG_RATIO, seed=SEED_BASE + i)

    if replay_ds is not None and len(replay_ds) > 0:
        if len(replay_ds) > temp_replay_cap:
            replay_ds = replay_ds.shuffle(seed=SEED_BASE + i).select(range(temp_replay_cap))

        temp_combined = concatenate_datasets([current_ds, replay_ds]).shuffle(seed=SEED_BASE + i)
        print(f"   Added replay buffer directly: {len(replay_ds)} examples")
    else:
        temp_combined = current_ds
        print("   No replay buffer yet.")

    combined_ds = temp_combined

    dpo_ds = combined_ds.map(prepare_example, remove_columns=combined_ds.column_names)

    sft_fraction = 0.20
    pos_ex = dpo_ds.filter(lambda ex: "Yes" in ex["chosen"])

    num_pos_available = len(pos_ex)
    oversample_factor = 1.5
    max_from_pos = int(num_pos_available * oversample_factor)
    max_relative = int(len(dpo_ds) * 0.15)
    n_sft_pos = min(max_from_pos, max_relative, num_pos_available, 2000)
    print(f"   Positive examples available: {num_pos_available} → selecting {n_sft_pos} pseudo-SFT")

    sft_examples = []
    for ex in pos_ex.shuffle(seed=SEED_BASE + i).select(range(n_sft_pos)):
        sft_examples.append({
            "prompt": ex["prompt"],
            "chosen": ex["chosen"],
            "rejected": ex["chosen"],
        })

    dpo_ds = concatenate_datasets([dpo_ds] + [Dataset.from_list(sft_examples)])
    dpo_ds = dpo_ds.shuffle(seed=SEED_BASE + i)
    print(f"   Added {len(sft_examples)} positive-only pseudo-SFT examples for regularization")

    early_stopping_callback = EarlyStoppingCallback(
        window_size=6,
        # window_size=3,
        threshold=0.003,
        # threshold=0.005,
        min_steps=10
        # min_steps=6
    )

    trainer = DPOTrainer(
        model=model,
        args=dpo_config,
        train_dataset=dpo_ds,
        tokenizer=tokenizer,
        callbacks=[early_stopping_callback]
    )

    trainer.train()

    if hasattr(early_stopping_callback, 'should_stop') and early_stopping_callback.should_stop:
        print(f"✓ Training stopped early at step {trainer.state.global_step}/{dpo_config.max_steps}")
    else:
        print(f"✓ Training completed all {dpo_config.max_steps} steps")

    if (i + 1) % 4 == 0:
        trainer.save_model(str(CHECKPOINT_DIR / f"month-{i+1}"))
        tokenizer.save_pretrained(str(CHECKPOINT_DIR / f"month-{i+1}"))
        print(f"Saved checkpoint → {CHECKPOINT_DIR / f'month-{i+1}'}")

    del trainer
    torch.cuda.empty_cache()
    gc.collect()

    mispred_eval_ds = create_balanced_sample(combined_ds, seed=SEED_BASE + i + 1000)
    print(f"   Eval set: {len(mispred_eval_ds)} rows")

    new_replay_ds, iteration_metrics = get_mispredicted_dataset(model, tokenizer, mispred_eval_ds)

    iteration_metrics["month_index"] = i + 1
    iteration_metrics["month_file"] = os.path.basename(csv_file)
    iteration_metrics["mispred_eval_ds_len"] = len(mispred_eval_ds)
    iteration_metrics["dpo_ds_len"] = len(dpo_ds)
    iteration_metrics["actual_steps"] = len(early_stopping_callback.loss_history) if hasattr(early_stopping_callback, 'loss_history') else 0
    metrics_history.append(iteration_metrics)

    print(f"   Misprediction rate: {len(new_replay_ds) / iteration_metrics['total']:.2%}")
    print(f"   Eval Metrics: TP={iteration_metrics['tp']} FN={iteration_metrics['fn']} FP={iteration_metrics['fp']} Prec={iteration_metrics['precision']:.3f} Rec={iteration_metrics['recall']:.3f} FPR={iteration_metrics.get('fpr', 0.0):.3f}")

    if prev_replay_size is not None:
        print(f"   Replay buffer trend: {prev_replay_size} → {len(new_replay_ds)}")
    prev_replay_size = len(new_replay_ds)

    prev_fn_examples = new_replay_ds.filter(lambda ex: ex["failure"] == 1)
    prev_fp_examples = new_replay_ds.filter(lambda ex: ex["failure"] == 0)

    print(f"   Raw mispredictions: {len(prev_fn_examples)} FN   {len(prev_fp_examples)} FP")

    replay_ds = create_balanced_replay(
        fn_examples=prev_fn_examples,
        fp_examples=prev_fp_examples,
        current_ds=current_ds,
        seed=SEED_BASE + i,
        max_size=MAX_REPLAY_SIZE,
        fallback_ratio=0.5,
        include_healthy_fallback=True
    )

    print(f"   Balanced replay buffer size: {len(replay_ds)}")

model.save_pretrained(str(MODEL_OUTPUT_DIR / "final_adapter"))
tokenizer.save_pretrained(str(MODEL_OUTPUT_DIR / "final_adapter"))
print("Final model saved successfully!")

with open(METRICS_JSON_PATH, "w") as f:
    json.dump(metrics_history, f, indent=2)

pd.DataFrame(metrics_history).to_csv(METRICS_CSV_PATH, index=False)

print(f"Final model saved to: {MODEL_OUTPUT_DIR / 'final_adapter'}")
print(f"Metrics history saved to: {METRICS_JSON_PATH}")
print(f"Metrics CSV saved to: {METRICS_CSV_PATH}")

In [ ]:
model_tuning_end_time = time.perf_counter()
elapsed_time = model_tuning_end_time - model_tuning_start_time
print(f"Model tuning elapsed time: {elapsed_time:.6f} seconds")

## Inspect Metrics

In [ ]:
for index, metrics in enumerate(metrics_history):
    print(
        f"Month: {index + 1} \t "
        f"Precision: {metrics['precision']:.5f} \t "
        f"Recall: {metrics['recall']:.5f} \t "
        f"FN: {metrics['fn']}"
    )

In [ ]:
if metrics_history:
    metrics_df = pd.DataFrame(metrics_history)

    if "month_file" in metrics_df.columns:
        metrics_df["month"] = metrics_df["month_file"].str.extract(r"(\d{4}[-_]?\d{2})")[0]
        metrics_df["month"] = metrics_df["month"].str.replace("_", "-", regex=False)
        metrics_df = metrics_df.sort_values("month")
        x_col = "month"
    else:
        x_col = "month_index"

    metrics_df.plot(
        x=x_col,
        y=["precision", "recall"],
        marker="o",
        figsize=(11, 6),
        grid=True,
        ylim=(0, 1.05),
        title="Tuning Precision and Recall",
        rot=45,
    )
    plt.tight_layout()
    plt.show()
else:
    print("metrics_history is empty.")